In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')
import random
from tqdm import tqdm
from scipy import stats

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


def set_seed(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


class TfidfModel:
    """TF-IDF model with your best pattern"""
    
    def __init__(self):
        self.best_token_pattern = r'(\b[A-Za-z_]\w*\b|[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+|[ \t\(\),;\{\}\[\]`"\'])'
        
    def preprocess(self, x):
        """Your preprocessing function"""
        return pd.Series(x).replace(r'\b([A-Za-z])\1+\b', '', regex=True)\
            .replace(r'\b[A-Za-z]\b', '', regex=True)
    
    def train(self, train_texts, train_labels, random_seed=42):
        """Train TF-IDF model"""
        transformer = FunctionTransformer(self.preprocess)
        vectorizer = TfidfVectorizer(
            token_pattern=self.best_token_pattern,
            max_features=5000,  
            lowercase=False
        )
        
        base_estimator = RandomForestClassifier(
            n_jobs=4,
            random_state=random_seed,
            n_estimators=200  
        )
        
        self.pipeline = Pipeline([
            ('preprocessing', transformer),
            ('vectorizer', vectorizer),
            ('clf', OneVsRestClassifier(base_estimator))
        ])
        
        best_params = {
            'clf__estimator__criterion': 'gini',
            'clf__estimator__max_features': 'log2',
            'clf__estimator__min_samples_split': 3,
            'clf__estimator__n_estimators': 200
        }
        self.pipeline.set_params(**best_params)
        
        self.pipeline.fit(train_texts, train_labels)
        
    def predict(self, texts):
        """Get predictions"""
        return self.pipeline.predict(texts)
    
    def predict_proba(self, texts):
        """Get probability predictions"""
        return self.pipeline.predict_proba(texts)


class weighted_average:
    """weighted_average fusion with optimal alpha search"""
    
    @staticmethod
    def weighted_average_fusion(tfidf_probs, codebert_probs, alpha):
        """Weighted average fusion"""
        fused_probs = alpha * tfidf_probs + (1 - alpha) * codebert_probs
        predictions = np.argmax(fused_probs, axis=1)
        return predictions, fused_probs
    
    @staticmethod
    def find_optimal_alpha(tfidf_probs, codebert_probs, true_labels):
        """Find optimal alpha using grid search"""
        best_accuracy = 0
        best_alpha = 0.5
        
        for alpha in np.arange(0, 1.01, 0.05):  
            predictions, _ = weighted_average.weighted_average_fusion(
                tfidf_probs, codebert_probs, alpha
            )
            accuracy = accuracy_score(true_labels, predictions)
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_alpha = alpha
        
        return best_alpha, best_accuracy


def load_and_prepare_reasonable_dataset(file_path, test_size=0.2, min_samples_per_author=50, max_authors=100):
    """
    Load Google Jam dataset with reasonable parameters
    """
    print("Loading dataset...")
    data = pd.read_csv(file_path)
    
    data = data.dropna(subset=['flines', 'username'])
    data['flines'] = data['flines'].astype(str)
    data = data[data['flines'].str.strip() != '']
    
    author_counts = data['username'].value_counts()
    
    valid_authors = author_counts[author_counts >= min_samples_per_author].index
    filtered_data = data[data['username'].isin(valid_authors)]
    
    if len(valid_authors) > max_authors:
        top_authors = author_counts.head(max_authors).index
        filtered_data = filtered_data[filtered_data['username'].isin(top_authors)]
    
    label_encoder = LabelEncoder()
    filtered_data['EncodedLabels'] = label_encoder.fit_transform(filtered_data['username'])
    num_classes = len(label_encoder.classes_)
    
    train_data, test_data = custom_stratified_split(filtered_data, test_size=test_size)
    
    return train_data, test_data, label_encoder, num_classes

def custom_stratified_split(data, test_size=0.2):
    """Custom stratified split that ensures proper distribution"""
    train_data = []
    test_data = []
    
    grouped = data.groupby('username')
    
    for author, group in grouped:
        group = group.sample(frac=1, random_state=42).reset_index(drop=True)
        
        n_test = max(1, int(len(group) * test_size))
        
        if len(group) - n_test < 2:
            n_test = max(1, len(group) - 2)
        
        test_samples = group.iloc[:n_test]
        train_samples = group.iloc[n_test:]
        
        test_data.append(test_samples)
        train_data.append(train_samples)
    
    train_data = pd.concat(train_data, ignore_index=True)
    test_data = pd.concat(test_data, ignore_index=True)
    
    train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
    test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)
    
    return train_data, test_data


class CodeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.df = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokens = self.tokenizer(
            row["flines"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": tokens["input_ids"].squeeze(0),
            "attention_mask": tokens["attention_mask"].squeeze(0),
            "labels": torch.tensor(row["EncodedLabels"], dtype=torch.long)
        }


class CodeBERTStage1(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(
            self.codebert.config.hidden_size,
            num_classes
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        token_embeddings = outputs.last_hidden_state
        attention_mask_exp = attention_mask.unsqueeze(-1)
        pooled_output = (token_embeddings * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

class CodeBERT_RI_Transformer(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.codebert = AutoModel.from_pretrained(
            "microsoft/codebert-base",
            output_hidden_states=True
        )
        hidden = self.codebert.config.hidden_size
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=8,
            batch_first=True
        )
        self.layer_transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )
        
        self.att_fc = nn.Linear(hidden, hidden)
        self.context_vector = nn.Parameter(torch.randn(hidden))
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        hidden_states = outputs.hidden_states[1:]
        attention_mask_exp = attention_mask.unsqueeze(-1)
        
        layerwise_embeddings = []
        for layer in hidden_states:
            pooled = (layer * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
            layerwise_embeddings.append(pooled)
        
        layer_sequence = torch.stack(layerwise_embeddings, dim=1)
        h = self.layer_transformer(layer_sequence)
        
        u = torch.tanh(self.att_fc(h))
        scores = torch.matmul(u, self.context_vector)
        alpha = torch.softmax(scores, dim=1)
        x_out = torch.sum(h * alpha.unsqueeze(-1), dim=1)
        
        return self.classifier(x_out)


def run_multi_seed_experiment(num_seeds=10):
    """Run multi-seed experiment with TF-IDF + CodeBERT fusion"""
    
    print("\n" + "="*80)
    print("MULTI-SEED weighted_average FUSION EXPERIMENT")
    print("="*80)
    
    SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555][:num_seeds]
    
    file_path = "/home/aman_swaraj/Downloads/Codelite/Other SE Tasks/Authorship_Attribution/gcj2020.csv"
    print("\nLoading dataset with Option 1 (Top 1000 authors)...")
    
    train_data, test_data, label_encoder, num_classes = load_and_prepare_reasonable_dataset(
        file_path, 
        test_size=0.2,
        min_samples_per_author=2,  
        max_authors=1000
    )
    
    print(f"\nDataset Statistics:")
    print(f"  Number of authors: {num_classes}")
    print(f"  Training samples: {len(train_data)}")
    print(f"  Test samples: {len(test_data)}")
    print(f"  Number of seeds: {num_seeds}")
    
    all_results = []
    
    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n{'='*60}")
        print(f"SEED {seed_idx+1}/{num_seeds}: {seed}")
        print(f"{'='*60}")
        
        set_seed(seed)
        
        print("\n1. Training TF-IDF model...")
        tfidf_model = TfidfModel()
        tfidf_model.train(train_data["flines"], train_data["EncodedLabels"], random_seed=seed)
        
        tfidf_preds = tfidf_model.predict(test_data["flines"])
        tfidf_probs = tfidf_model.predict_proba(test_data["flines"])
        tfidf_accuracy = accuracy_score(test_data["EncodedLabels"], tfidf_preds)
        print(f"   TF-IDF Accuracy: {tfidf_accuracy:.4f}")
        
        print("\n2. Training CodeBERT two-stage model...")
        
        tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        
        train_dataset = CodeDataset(train_data, tokenizer)
        test_dataset = CodeDataset(test_data, tokenizer)
        
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
        
        stage1_model = CodeBERTStage1(num_classes).to(device)
        optimizer = optim.AdamW(stage1_model.parameters(), lr=2e-5, weight_decay=0.01)
        criterion = nn.CrossEntropyLoss()
        
        stage1_model.train()
        epochs_stage1 = 3
        
        for epoch in range(epochs_stage1):
            total_loss = 0
            for batch in tqdm(train_loader, desc=f"Stage 1 Epoch {epoch+1}", leave=False):
                optimizer.zero_grad()
                logits = stage1_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
                loss = criterion(logits, batch["labels"].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(stage1_model.parameters(), max_norm=1.0)
                optimizer.step()
                total_loss += loss.item()
        
        torch.save(stage1_model.codebert.state_dict(), 
                  f"/home/aman_swaraj/Downloads/Codelite/codebert_stage1_seed{seed}.pt")
        
        stage2_model = CodeBERT_RI_Transformer(num_classes).to(device)
        stage2_model.codebert.load_state_dict(
            torch.load(f"/home/aman_swaraj/Downloads/Codelite/codebert_stage1_seed{seed}.pt")
        )
        
        for param in stage2_model.codebert.parameters():
            param.requires_grad = False
        
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, stage2_model.parameters()),
            lr=1e-4,
            weight_decay=0.01
        )
        
        stage2_model.train()
        epochs_stage2 = 5
        
        for epoch in range(epochs_stage2):
            total_loss = 0
            for batch in tqdm(train_loader, desc=f"Stage 2 Epoch {epoch+1}", leave=False):
                optimizer.zero_grad()
                logits = stage2_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
                loss = criterion(logits, batch["labels"].to(device))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, stage2_model.parameters()), 
                    max_norm=1.0
                )
                optimizer.step()
                total_loss += loss.item()
        
        stage2_model.eval()
        all_codebert_preds = []
        all_codebert_probs = []
        all_labels = []
        
        with torch.no_grad():
            for batch in test_loader:
                logits = stage2_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device)
                )
                probs = torch.softmax(logits, dim=1)
                preds = torch.argmax(logits, dim=1)
                
                all_codebert_preds.extend(preds.cpu().numpy())
                all_codebert_probs.extend(probs.cpu().numpy())
                all_labels.extend(batch["labels"].cpu().numpy())
        
        codebert_accuracy = accuracy_score(all_labels, all_codebert_preds)
        codebert_probs = np.array(all_codebert_probs)
        print(f"   CodeBERT Accuracy: {codebert_accuracy:.4f}")
        
        print("\n3. Applying weighted_average fusion...")
        
        optimal_alpha, fusion_accuracy = weighted_average.find_optimal_alpha(
            tfidf_probs, codebert_probs, test_data["EncodedLabels"]
        )
        
        fusion_preds, fusion_probs = weighted_average.weighted_average_fusion(
            tfidf_probs, codebert_probs, optimal_alpha
        )
        
        best_single_accuracy = max(tfidf_accuracy, codebert_accuracy)
        improvement = fusion_accuracy - best_single_accuracy
        
        model_agreement = np.mean(tfidf_preds == np.array(all_codebert_preds))
        
        seed_results = {
            'seed': seed,
            'tfidf_accuracy': tfidf_accuracy,
            'codebert_accuracy': codebert_accuracy,
            'best_single_accuracy': best_single_accuracy,
            'optimal_alpha': optimal_alpha,
            'fusion_accuracy': fusion_accuracy,
            'absolute_improvement': improvement,
            'relative_improvement': improvement / (1 - best_single_accuracy) * 100 if best_single_accuracy < 1 else 0,
            'model_agreement': model_agreement,
            'fusion_beats_both': fusion_accuracy > best_single_accuracy
        }
        
        all_results.append(seed_results)
        
        print(f"   Optimal alpha: {optimal_alpha:.3f}")
        print(f"   weighted_average Fusion Accuracy: {fusion_accuracy:.4f}")
        print(f"   Improvement over best single: {improvement:.4f}")
        print(f"   Model Agreement: {model_agreement:.2%}")
        print(f"   Fusion beats best single: {'YES' if seed_results['fusion_beats_both'] else 'NO'}")
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS ANALYSIS")
    print("="*80)
    
    results_df = pd.DataFrame(all_results)
    
    print("\nAverage Performance Across Seeds:")
    print("-" * 60)
    
    summary_stats = {
        'Metric': ['TF-IDF Accuracy', 'CodeBERT Accuracy', 'Best Single Model', 
                  'weighted_average Fusion', 'Optimal Alpha', 'Absolute Improvement',
                  'Relative Improvement (%)', 'Model Agreement', 'Fusion Success Rate (%)'],
        'Mean': [
            results_df['tfidf_accuracy'].mean(),
            results_df['codebert_accuracy'].mean(),
            results_df['best_single_accuracy'].mean(),
            results_df['fusion_accuracy'].mean(),
            results_df['optimal_alpha'].mean(),
            results_df['absolute_improvement'].mean(),
            results_df['relative_improvement'].mean(),
            results_df['model_agreement'].mean(),
            results_df['fusion_beats_both'].mean() * 100
        ],
        'Std': [
            results_df['tfidf_accuracy'].std(),
            results_df['codebert_accuracy'].std(),
            results_df['best_single_accuracy'].std(),
            results_df['fusion_accuracy'].std(),
            results_df['optimal_alpha'].std(),
            results_df['absolute_improvement'].std(),
            results_df['relative_improvement'].std(),
            results_df['model_agreement'].std(),
            0
        ]
    }
    
    summary_df = pd.DataFrame(summary_stats)
    print(summary_df.to_string(index=False))
    
    print("\n" + "-" * 60)
    print("STATISTICAL SIGNIFICANCE TEST")
    print("-" * 60)
    
    improvements = results_df['absolute_improvement']
    t_stat, p_value = stats.ttest_1samp(improvements, 0)
    
    print(f"Mean absolute improvement: {improvements.mean():.4f} ± {improvements.std():.4f}")
    print(f"t-statistic: {t_stat:.4f}, p-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("✓ The improvement is statistically significant (p < 0.05)")
        significance = "statistically significant"
    else:
        print("✗ The improvement is not statistically significant")
        significance = "not statistically significant"
    
    success_rate = results_df['fusion_beats_both'].mean() * 100
    print(f"\nFusion Success Rate: {success_rate:.1f}% ({results_df['fusion_beats_both'].sum()}/{num_seeds} seeds)")
    
    print(f"\nOptimal Alpha Analysis:")
    print(f"  Mean: {results_df['optimal_alpha'].mean():.3f}")
    print(f"  Std: {results_df['optimal_alpha'].std():.3f}")
    print(f"  Range: [{results_df['optimal_alpha'].min():.3f}, {results_df['optimal_alpha'].max():.3f}]")
    print(f"  Interpretation: TF-IDF weight {results_df['optimal_alpha'].mean()*100:.1f}%, CodeBERT weight {(1-results_df['optimal_alpha'].mean())*100:.1f}%")
    
    print("\n" + "="*60)
    print("SAVING RESULTS")
    print("="*60)
    
    results_path = "/home/aman_swaraj/Downloads/Codelite/authorship_fusion_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Detailed results saved to: {results_path}")
    
    summary_path = "/home/aman_swaraj/Downloads/Codelite/authorship_fusion_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary statistics saved to: {summary_path}")
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        
        print("\nGenerating visualizations...")
        
        plt.figure(figsize=(12, 6))
        
        seeds_list = [f"Seed {s}" for s in SEEDS]
        x = np.arange(num_seeds)
        width = 0.2
        
        plt.bar(x - width*1.5, results_df['tfidf_accuracy'], width, 
                label='TF-IDF', color='#FF6B6B', alpha=0.8)
        plt.bar(x - width*0.5, results_df['codebert_accuracy'], width, 
                label='CodeBERT', color='#4ECDC4', alpha=0.8)
        plt.bar(x + width*0.5, results_df['best_single_accuracy'], width, 
                label='Best Single', color='#FFD166', alpha=0.8)
        plt.bar(x + width*1.5, results_df['fusion_accuracy'], width, 
                label='weighted_average Fusion', color='#06D6A0', alpha=0.8)
        
        plt.xlabel('Random Seed')
        plt.ylabel('Accuracy')
        plt.title(f'Authorship Attribution: weighted_average Fusion Across {num_seeds} Seeds')
        plt.xticks(x, seeds_list)
        plt.legend()
        plt.grid(True, alpha=0.3, linestyle='--')
        
        for i in range(num_seeds):
            improvement = results_df['absolute_improvement'].iloc[i]
            if improvement > 0:
                plt.text(i + width*1.5, results_df['fusion_accuracy'].iloc[i] + 0.005,
                        f"+{improvement:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('authorship_fusion_performance.png', dpi=300, bbox_inches='tight')
        print("  Saved: authorship_fusion_performance.png")
        
        plt.figure(figsize=(10, 6))
        
        plt.subplot(1, 2, 1)
        improvements_pct = results_df['relative_improvement']
        colors = ['#06D6A0' if imp > 0 else '#EF476F' for imp in improvements_pct]
        bars = plt.bar(range(1, num_seeds + 1), improvements_pct, color=colors)
        plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
        plt.xlabel('Seed')
        plt.ylabel('Relative Improvement (%)')
        plt.title('Fusion Improvement by Seed')
        plt.grid(True, alpha=0.3, linestyle='--')
        
        for i, (bar, imp) in enumerate(zip(bars, improvements_pct)):
            plt.text(bar.get_x() + bar.get_width()/2, 
                    imp + (0.5 if imp >= 0 else -1.5),
                    f'{imp:.1f}%', ha='center', va='bottom' if imp >= 0 else 'top')
        
        plt.subplot(1, 2, 2)
        plt.hist(results_df['optimal_alpha'], bins=5, alpha=0.7, 
                 color='#118AB2', edgecolor='black')
        plt.axvline(x=results_df['optimal_alpha'].mean(), color='red', 
                    linestyle='--', linewidth=2, label=f'Mean: {results_df["optimal_alpha"].mean():.3f}')
        plt.xlabel('Optimal Alpha (TF-IDF weight)')
        plt.ylabel('Frequency')
        plt.title('Distribution of Optimal Alpha Values')
        plt.legend()
        plt.grid(True, alpha=0.3, linestyle='--')
        
        plt.tight_layout()
        plt.savefig('authorship_fusion_improvements.png', dpi=300, bbox_inches='tight')
        print("  Saved: authorship_fusion_improvements.png")
        
        plt.show()
        
    except ImportError:
        print("\nVisualization libraries not available. Skipping plots.")
    



if __name__ == "__main__":
    print("Authorship Attribution with TF-IDF + CodeBERT weighted_average Fusion")
    print("="*80)
    
    num_seeds = 10  
    results_df, summary_df = run_multi_seed_experiment(num_seeds=num_seeds)
    
    print("\n" + "="*80)
    print("FINAL SUMMARY")
    print("="*80)
    print(f"Dataset: Google Code Jam 2008 (Top 50 authors)")
    print(f"Number of seeds evaluated: {num_seeds}")
    print(f"Average TF-IDF Accuracy: {results_df['tfidf_accuracy'].mean():.4f}")
    print(f"Average CodeBERT Accuracy: {results_df['codebert_accuracy'].mean():.4f}")
    print(f"Average Fusion Accuracy: {results_df['fusion_accuracy'].mean():.4f}")
    print(f"Average Improvement: {results_df['absolute_improvement'].mean():.4f}")
    print(f"Optimal α: {results_df['optimal_alpha'].mean():.3f}")